# Core Concepts of LangGraph

## Tutorials Covered:
1. State Management
2. Nodes and Edges
3. Conditional Logic
4. Parallel Processing

## 1. State Management

Learning objectives:
- Understand how LangGraph handles state in multi-step processes
- Learn about different state management strategies
- Implement state management in your graphs

In [ ]:
# Tutorial 1: State Management

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

print("=== State Management in LangGraph ===\n")

# 1. Understanding State
print("1. STATE represents the memory of your application")
print("   It persists data across different node executions\n")

# Define a comprehensive state structure
class StateManagementState(TypedDict):
    # Basic data types
    counter: int
    message: str
    is_active: bool
    
    # Collections
    items: list[str]
    metadata: dict
    
    # Tracking variables
    execution_log: list[str]
    errors: list[str]

# Node to initialize state
def initialize_state_node(state):
    log_entry = f"Initializing state with counter={state.get('counter', 0)}"
    print(log_entry)
    return {
        "counter": 0,
        "message": "Initial message",
        "is_active": True,
        "items": [],
        "metadata": {"created_by": "initialize_node", "version": 1},
        "execution_log": [log_entry],
        "errors": []
    }

# Node to update state
def update_state_node(state):
    log_entry = f"Updating state: counter {state['counter']} -> {state['counter'] + 1}"
    print(log_entry)
    
    updated_items = state['items'] + [f"item_{state['counter'] + 1}"]
    updated_metadata = state['metadata'].copy()
    updated_metadata["last_updated"] = state['counter'] + 1
    
    return {
        "counter": state['counter'] + 1,
        "message": f"Updated message at counter {state['counter'] + 1}",
        "items": updated_items,
        "metadata": updated_metadata,
        "execution_log": state['execution_log'] + [log_entry]
    }

# Node to demonstrate state reduction
def reduce_state_node(state):
    log_entry = f"Reducing state with {len(state['items'])} items"
    print(log_entry)
    
    # Demonstrates how state can be aggregated
    summary = f"Processed {state['counter']} items: {', '.join(state['items'])}"
    
    return {
        "message": summary,
        "execution_log": state['execution_log'] + [log_entry]
    }

# Create the state management graph
builder = StateGraph(StateManagementState)

# Add nodes
builder.add_node("initialize", initialize_state_node)
builder.add_node("update", update_state_node)
builder.add_node("reduce", reduce_state_node)

# Connect nodes
builder.add_edge("__start__", "initialize")
builder.add_edge("initialize", "update")
builder.add_edge("update", "reduce")
builder.add_edge("reduce", "__end__")

# Compile the graph
state_management_graph = builder.compile()

print("2. Executing state management graph:\n")

# Initial state
initial_state = {
    "counter": -1,  # Will be overridden by initialize node
    "message": "",
    "is_active": False,
    "items": [],
    "metadata": {},
    "execution_log": ["Starting state management tutorial"],
    "errors": []
}

# Run the graph
result = state_management_graph.invoke(initial_state)

print(f"\nFinal state: {result}\n")

print("3. Key concepts of state management:")
print("   - State persists across node executions")
print("   - Each node can read and modify parts of the state")
print("   - State structure is defined upfront using TypedDict")
print("   - Updates are merged into the existing state")

## 2. Nodes and Edges

Learning objectives:
- Create and connect different components in your graph
- Understand different types of nodes
- Define relationships between nodes using edges

In [ ]:
# Tutorial 2: Nodes and Edges

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

print("=== Nodes and Edges in LangGraph ===\n")

# Define a state for our demonstration
class NodesAndEdgesState(TypedDict):
    value: int
    operation_log: list[str]
    current_operation: str

# Different types of nodes

def start_node(state):
    print("Executing START node")
    return {"value": 0, "current_operation": "start", "operation_log": ["Started"]}

def calculation_node(state):
    print(f"Executing CALCULATION node with value {state['value']}")
    new_value = state['value'] + 10
    return {
        "value": new_value,
        "current_operation": "calculation",
        "operation_log": state['operation_log'] + [f"Added 10 to get {new_value}"]
    }

def transformation_node(state):
    print(f"Executing TRANSFORMATION node with value {state['value']}")
    new_value = state['value'] * 2
    return {
        "value": new_value,
        "current_operation": "transformation",
        "operation_log": state['operation_log'] + [f"Multiplied by 2 to get {new_value}"]
    }

def validation_node(state):
    print(f"Executing VALIDATION node with value {state['value']}")
    status = "valid" if state['value'] > 0 else "invalid"
    return {
        "current_operation": "validation",
        "operation_log": state['operation_log'] + [f"Value {state['value']} is {status}"]
    }

def aggregation_node(state):
    print(f"Executing AGGREGATION node with value {state['value']}")
    return {
        "current_operation": "aggregation",
        "operation_log": state['operation_log'] + [f"Final value: {state['value']}"]
    }

print("1. NODES represent individual computational steps")
print("   Each node is a function that takes the current state and returns updates\n")

print("2. EDGES define the flow between nodes")
print("   They specify which node executes after another\n")

# Create the graph
builder = StateGraph(NodesAndEdgesState)

# Add nodes
builder.add_node("start", start_node)
builder.add_node("calculate", calculation_node)
builder.add_node("transform", transformation_node)
builder.add_node("validate", validation_node)
builder.add_node("aggregate", aggregation_node)

# Add edges - demonstrating different connection patterns
builder.add_edge("start", "calculate")      # Linear: start -> calculate
builder.add_edge("calculate", "transform")   # Linear: calculate -> transform
builder.add_edge("transform", "validate")    # Linear: transform -> validate
builder.add_edge("validate", "aggregate")    # Linear: validate -> aggregate
builder.add_edge("aggregate", "__end__")     # End the graph

# Set entry point
builder.set_entry_point("start")

# Compile the graph
nodes_edges_graph = builder.compile()

print("3. Executing the nodes and edges graph:\n")

# Run the graph
result = nodes_edges_graph.invoke({"value": 5, "operation_log": [], "current_operation": "initial"})

print(f"\nFinal result: {result}\n")

print("4. Key concepts of nodes and edges:")
print("   - Nodes are stateful functions that perform specific tasks")
print("   - Edges connect nodes and define execution order")
print("   - Data flows through the graph via the shared state")
print("   - Multiple edges can converge into a single node")

## 3. Conditional Logic

Learning objectives:
- Implement decision-making in your graphs
- Use conditional edges for branching logic
- Handle different execution paths

In [ ]:
# Tutorial 3: Conditional Logic

from langgraph.graph import StateGraph, END
from typing import TypedDict

print("=== Conditional Logic in LangGraph ===\n")

# Define state for conditional logic
class ConditionalLogicState(TypedDict):
    value: int
    category: str
    processing_path: str
    history: list[str]

# Define nodes for different processing paths

def input_classifier_node(state):
    print(f"Classifying input value: {state['value']}")
    
    if state['value'] < 0:
        category = "negative"
    elif state['value'] == 0:
        category = "zero"
    elif state['value'] <= 10:
        category = "small_positive"
    elif state['value'] <= 100:
        category = "medium_positive"
    else:
        category = "large_positive"
        
    return {
        "category": category,
        "history": state['history'] + [f"Classified {state['value']} as {category}"]
    }

def negative_processor_node(state):
    print(f"Processing negative value: {state['value']}")
    new_value = abs(state['value'])  # Make positive
    return {
        "value": new_value,
        "processing_path": "negative_to_positive",
        "history": state['history'] + [f"Converted {state['value']} to {new_value}"]
    }

def zero_handler_node(state):
    print(f"Handling zero value: {state['value']}")
    new_value = 1  # Set to 1
    return {
        "value": new_value,
        "processing_path": "zero_to_one",
        "history": state['history'] + [f"Changed zero to {new_value}"]
    }

def small_number_processor_node(state):
    print(f"Processing small positive value: {state['value']}")
    new_value = state['value'] * 2
    return {
        "value": new_value,
        "processing_path": "small_doubled",
        "history": state['history'] + [f"Doubled {state['value']} to {new_value}"]
    }

def medium_number_processor_node(state):
    print(f"Processing medium positive value: {state['value']}")
    new_value = state['value'] // 2
    return {
        "value": new_value,
        "processing_path": "medium_halved",
        "history": state['history'] + [f"Halved {state['value']} to {new_value}"]
    }

def large_number_processor_node(state):
    print(f"Processing large positive value: {state['value']}")
    new_value = int(state['value'] ** 0.5)  # Square root
    return {
        "value": new_value,
        "processing_path": "large_sqrt",
        "history": state['history'] + [f"Took square root of {state['value']} to get {new_value}"]
    }

def aggregator_node(state):
    print(f"Aggregating results for final value: {state['value']}")
    return {
        "history": state['history'] + [f"Final value after processing: {state['value']}"]
    }

# Function to determine the next node based on state
# This is the conditional edge logic
def routing_function(state):
    print(f"Routing based on category: {state['category']}")
    
    if state['category'] == 'negative':
        return 'negative_processor'
    elif state['category'] == 'zero':
        return 'zero_handler'
    elif state['category'] == 'small_positive':
        return 'small_number_processor'
    elif state['category'] == 'medium_positive':
        return 'medium_number_processor'
    elif state['category'] == 'large_positive':
        return 'large_number_processor'
    else:
        return 'aggregator'  # fallback

print("1. CONDITIONAL LOGIC allows dynamic routing based on state values")
print("   The routing function determines the next node based on conditions\n")

# Create the graph with conditional logic
builder = StateGraph(ConditionalLogicState)

# Add all nodes
builder.add_node("classifier", input_classifier_node)
builder.add_node("negative_processor", negative_processor_node)
builder.add_node("zero_handler", zero_handler_node)
builder.add_node("small_number_processor", small_number_processor_node)
builder.add_node("medium_number_processor", medium_number_processor_node)
builder.add_node("large_number_processor", large_number_processor_node)
builder.add_node("aggregator", aggregator_node)

# Add the starting edge
builder.add_edge("__start__", "classifier")

# Add conditional edge from classifier to route to appropriate processor
builder.add_conditional_edges(
    "classifier",
    routing_function,
    {
        "negative_processor": "negative_processor",
        "zero_handler": "zero_handler",
        "small_number_processor": "small_number_processor",
        "medium_number_processor": "medium_number_processor",
        "large_number_processor": "large_number_processor"
    }
)

# Add edges from all processors to aggregator
builder.add_edge("negative_processor", "aggregator")
builder.add_edge("zero_handler", "aggregator")
builder.add_edge("small_number_processor", "aggregator")
builder.add_edge("medium_number_processor", "aggregator")
builder.add_edge("large_number_processor", "aggregator")

# Add final edge to end
builder.add_edge("aggregator", "__end__")

# Compile the graph
conditional_graph = builder.compile()

print("2. Executing the conditional logic graph with different inputs:\n")

# Test with different values to trigger different branches
test_values = [-5, 0, 5, 50, 200]

for i, test_val in enumerate(test_values):
    print(f"--- Test {i+1}: Input value {test_val} ---")
    result = conditional_graph.invoke({
        "value": test_val,
        "category": "",
        "processing_path": "",
        "history": [f"Started with {test_val}"]
    })
    print(f"Result: {result['value']}, Path: {result['processing_path']}, History: {result['history'][-2:]})
    print()

print("3. Key concepts of conditional logic:")
print("   - Conditional edges route execution based on state conditions")
print("   - Routing functions evaluate the state and return the next node")
print("   - Different paths can be taken depending on the data")
print("   - This enables complex, dynamic workflows")

## 4. Parallel Processing

Learning objectives:
- Run multiple operations simultaneously in LangGraph
- Understand concurrent execution patterns
- Implement parallel workflows

In [ ]:
# Tutorial 4: Parallel Processing

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import asyncio
import time

print("=== Parallel Processing in LangGraph ===\n")

# Define state for parallel processing
class ParallelProcessingState(TypedDict):
    input_data: list[int]
    results: dict
    processing_times: dict
    status: str

# Define nodes for parallel operations

def data_splitter_node(state):
    print(f"Splitting data for parallel processing: {state['input_data']}")
    
    # Split the input data for different parallel processes
    positive_nums = [x for x in state['input_data'] if x > 0]
    negative_nums = [x for x in state['input_data'] if x < 0]
    zeros = [x for x in state['input_data'] if x == 0]
    
    return {
        "results": {
            "original": state['input_data'],
            "positive": positive_nums,
            "negative": negative_nums,
            "zeros": zeros
        },
        "status": "splitting_complete"
    }

def compute_squares_async(data):
    # Simulate async computation
    time.sleep(0.5)  # Simulate some processing time
    squares = [x**2 for x in data]
    return squares

def compute_cubes_async(data):
    # Simulate async computation
    time.sleep(0.7)  # Simulate some processing time
    cubes = [x**3 for x in data]
    return cubes

def compute_square_roots_async(data):
    # Simulate async computation
    time.sleep(0.6)  # Simulate some processing time
    import math
    sqrt_vals = [math.sqrt(abs(x)) for x in data]
    return sqrt_vals

def positive_processor_node(state):
    print(f"Processing positive numbers in parallel: {state['results']['positive']}")
    start_time = time.time()
    
    # Simulate async processing of positive numbers
    squares = compute_squares_async(state['results']['positive'])
    processing_time = time.time() - start_time
    
    return {
        "results": {**state['results'], "positive_squares": squares},
        "processing_times": {**state.get('processing_times', {}), "positive_processing": processing_time},
        "status": "positive_processed"
    }

def negative_processor_node(state):
    print(f"Processing negative numbers in parallel: {state['results']['negative']}")
    start_time = time.time()
    
    # Simulate async processing of negative numbers
    cubes = compute_cubes_async(state['results']['negative'])
    processing_time = time.time() - start_time
    
    return {
        "results": {**state['results'], "negative_cubes": cubes},
        "processing_times": {**state.get('processing_times', {}), "negative_processing": processing_time},
        "status": "negative_processed"
    }

def zero_processor_node(state):
    print(f"Processing zeros in parallel: {state['results']['zeros']}")
    start_time = time.time()
    
    # Simulate async processing of zeros
    sqrt_vals = compute_square_roots_async(state['results']['zeros'])
    processing_time = time.time() - start_time
    
    return {
        "results": {**state['results'], "zero_sqrt": sqrt_vals},
        "processing_times": {**state.get('processing_times', {}), "zero_processing": processing_time},
        "status": "zero_processed"
    }

def aggregator_node(state):
    print(f"Aggregating parallel processing results")
    
    total_processing_time = sum(state.get('processing_times', {}).values())
    
    # Create a summary of all results
    summary = {
        "total_inputs": len(state['results']['original']),
        "positive_count": len(state['results']['positive']),
        "negative_count": len(state['results']['negative']),
        "zero_count": len(state['results']['zeros']),
        "positive_squares": state['results'].get('positive_squares', []),
        "negative_cubes": state['results'].get('negative_cubes', []),
        "zero_sqrt": state['results'].get('zero_sqrt', [])
    }
    
    return {
        "results": {**state['results'], "summary": summary},
        "processing_times": {**state['processing_times'], "total": total_processing_time},
        "status": "complete"
    }

print("1. PARALLEL PROCESSING allows multiple operations to run simultaneously")
print("   Different nodes can process data concurrently based on the state\n")

# Create the graph with parallel processing
builder = StateGraph(ParallelProcessingState)

# Add all nodes
builder.add_node("splitter", data_splitter_node)
builder.add_node("positive_processor", positive_processor_node)
builder.add_node("negative_processor", negative_processor_node)
builder.add_node("zero_processor", zero_processor_node)
builder.add_node("aggregator", aggregator_node)

# Connect nodes
builder.add_edge("__start__", "splitter")

# From splitter, fan out to all processors (parallel execution)
builder.add_edge("splitter", "positive_processor")
builder.add_edge("splitter", "negative_processor")
builder.add_edge("splitter", "zero_processor")

# From all processors, converge to aggregator
builder.add_edge("positive_processor", "aggregator")
builder.add_edge("negative_processor", "aggregator")
builder.add_edge("zero_processor", "aggregator")

# End after aggregation
builder.add_edge("aggregator", "__end__")

# Compile the graph
parallel_graph = builder.compile()

print("2. Executing the parallel processing graph:\n")

# Run the graph with sample data
sample_data = [1, 2, 3, -1, -2, -3, 0, 0, 4, 5]
start_time = time.time()

result = parallel_graph.invoke({
    "input_data": sample_data,
    "results": {},
    "processing_times": {},
    "status": "initial"
})

execution_time = time.time() - start_time

print(f"\nFinal result: {result['results']['summary']}")
print(f"Processing times: {result['processing_times']}")
print(f"Total execution time: {execution_time:.2f}s")

print("\n3. Key concepts of parallel processing:")
print("   - Multiple nodes can execute simultaneously")
print("   - Fan-out pattern splits work across parallel paths")
print("   - Fan-in pattern aggregates results from parallel paths")
print("   - Parallel processing can significantly reduce execution time")